In [1]:
# import essential library for analysis
# --------------------------------------

import pandas as pd
import numpy as np

In [2]:
# load data into the df
# -----------------------

df=pd.read_csv("C:/indan-startup-ecosystem/00_Data/raw_data/startup_funding.csv")
df.head()

,Sr No,Date dd/mm/yyyy,Startup Name,Industry Vertical,SubVertical,City Location,Investors Name,InvestmentnType,Amount in USD,Remarks
0,1,09/01/2020,BYJU’S,E-Tech,E-learning,Bengaluru,Tiger Global Management,Private Equity Round,"20,00,00,000",NaN
1,2,13/01/2020,Shuttl,Transportation,App based shuttle service,Gurgaon,Susquehanna Growth Equity,Series C,"80,48,394",NaN
2,3,09/01/2020,Mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,Sequoia Capital India,Series B,"1,83,58,860",NaN
3,4,02/01/2020,https://www.wealthbucket.in/,FinTech,Online Investment,New Delhi,Vinod Khatumal,Pre-series A,"30,00,000",NaN
4,5,02/01/2020,Fashor,Fashion and Apparel,Embroiled Clothes For Women,Mumbai,Sprout Venture Partners,Seed Round,"18,00,000",NaN


In [4]:
# check the basic information about data
# ----------------------------------------

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3044 entries, 0 to 3043
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Sr No              3044 non-null   int64 
 1   Date dd/mm/yyyy    3044 non-null   object
 2   Startup Name       3044 non-null   object
 3   Industry Vertical  2873 non-null   object
 4   SubVertical        2108 non-null   object
 5   City  Location     2864 non-null   object
 6   Investors Name     3020 non-null   object
 7   InvestmentnType    3040 non-null   object
 8   Amount in USD      2084 non-null   object
 9   Remarks            419 non-null    object
dtypes: int64(1), object(9)
memory usage: 237.9+ KB


In [5]:
# change column name from Date dd/mm/yyyy to Date
# -------------------------------------------------

df=df.rename(columns={'Date dd/mm/yyyy' : 'Date'})

# ==============
# show top rows 
# ==============
df.head()

,Sr No,Date,Startup Name,Industry Vertical,SubVertical,City Location,Investors Name,InvestmentnType,Amount in USD,Remarks
0,1,09/01/2020,BYJU’S,E-Tech,E-learning,Bengaluru,Tiger Global Management,Private Equity Round,"20,00,00,000",NaN
1,2,13/01/2020,Shuttl,Transportation,App based shuttle service,Gurgaon,Susquehanna Growth Equity,Series C,"80,48,394",NaN
2,3,09/01/2020,Mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,Sequoia Capital India,Series B,"1,83,58,860",NaN
3,4,02/01/2020,https://www.wealthbucket.in/,FinTech,Online Investment,New Delhi,Vinod Khatumal,Pre-series A,"30,00,000",NaN
4,5,02/01/2020,Fashor,Fashion and Apparel,Embroiled Clothes For Women,Mumbai,Sprout Venture Partners,Seed Round,"18,00,000",NaN


In [6]:
# show all date records where date in dd/mmYY.
# --------------------------------------------
mask=df['Date'].str.count('/') == 1

# convert rows to correct format (dd/mmYY --> dd/mm/YY)
# -------------------------------------------------------
df.loc[mask, 'Date'] = df.loc[mask, 'Date'].str.replace(r'(\d{2})(\d{4})$', r'\1/\2', regex=True)

# # change the datatype from str to datetime
# -------------------------------------------
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')

### ================
## Date Column Issue
### ================
#### issue in the column, some values are like dd/mmYY format so i first convert the format to dd/mm/YY then i did into datetime datatype but pandas datetime function understand that mm/dd/YY but my data in date column have this type dd/mm/YY values, so i use dayfirst=True means tell the pandas function that day is first, i use errors='coerce' means it convert values into nan, whose create problem and it save the cleaning pipeline.

In [7]:
# check null values into dataset 
# -------------------------------
df['Date'].isna().sum()

np.int64(7)

In [8]:
# check missing value 
# --------------------

df[df['Date'].isna()]

,Sr No,Date,Startup Name,Industry Vertical,SubVertical,City Location,Investors Name,InvestmentnType,Amount in USD,Remarks
2571,2572,NaT,HandyHome,Hyperlocal Electronics repair Service,NaN,Mumbai,"Bessemer Ventures, Kae Capital",Seed Funding,"6,30,000",NaN
2606,2607,NaT,\\xc2\\xa0Infinity Assurance,\\xc2\\xa0Warranty Programs Service Administra...,NaN,\\xc2\\xa0New Delhi,\\xc2\\xa0Indian Angel Network,Seed Funding,"\\xc2\\xa0600,000",NaN
2775,2776,NaT,Mobiefit,Mobile Fitness App,NaN,Goa,Prototyze,Seed Funding,NaN,NaN
2776,2777,NaT,Plancess,Online Education Platform,NaN,Mumbai,"Rajendra Gogri,\\xc2\\xa0 Chandrakant Gogri",Seed Funding,"20,00,000",NaN
2831,2832,NaT,The Porter,Logistics Tech,NaN,Mumbai,Kae Capital,Seed Funding,"5,00,000",NaN
3011,3012,NaT,Wishberry,NaN,NaN,NaN,"Rajan Anandan, Shankar Mahadevan, Deep Kalra &...",Seed Funding,"6,50,000",NaN
3029,3030,NaT,Corporate360,NaN,NaN,NaN,Group of Angel Investors,Seed Funding,"2,00,000",NaN


In [9]:
# drop null rows of the date column
# ----------------------------------

df = df.dropna(subset=['Date'])

In [10]:
def clean_amount(value):
    if pd.isna(value) or value == 'N/A':
        return np.nan
    
    # Step 1: Convert to string
    val_str = str(value)
    
    # Step 2: Remove non-breaking spaces aur any other non-numeric chars except comma
    val_str = val_str.replace('\xc2\xa0', '').replace('$', '').strip()
    
    # Step 3: Remove commas (thousands separators)
    val_str = val_str.replace(',', '')
    
    # Step 4: Convert to float/int
    try:
        return float(val_str)
    except ValueError:
        return np.nan

# Apply the function
df['Amount'] = df['Amount in USD'].apply(clean_amount)

# Drop the original column
df = df.drop('Amount in USD', axis=1)

# Check result
print(df['Amount'].head(10))

0    200000000.0
1      8048394.0
2     18358860.0
3      3000000.0
4      1800000.0
5      9000000.0
6    150000000.0
7      6000000.0
8     70000000.0
9     50000000.0
Name: Amount, dtype: float64


In [11]:
# check outlieres
# ----------------

df.head()

,Sr No,Date,Startup Name,Industry Vertical,SubVertical,City Location,Investors Name,InvestmentnType,Remarks,Amount
0,1,2020-01-09,BYJU’S,E-Tech,E-learning,Bengaluru,Tiger Global Management,Private Equity Round,NaN,200000000.0
1,2,2020-01-13,Shuttl,Transportation,App based shuttle service,Gurgaon,Susquehanna Growth Equity,Series C,NaN,8048394.0
2,3,2020-01-09,Mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,Sequoia Capital India,Series B,NaN,18358860.0
3,4,2020-01-02,https://www.wealthbucket.in/,FinTech,Online Investment,New Delhi,Vinod Khatumal,Pre-series A,NaN,3000000.0
4,5,2020-01-02,Fashor,Fashion and Apparel,Embroiled Clothes For Women,Mumbai,Sprout Venture Partners,Seed Round,NaN,1800000.0


In [12]:
# Convert INR to USD (1 USD = 65 INR)
df['Amount'] = df['Amount'] / 65

df.head()

,Sr No,Date,Startup Name,Industry Vertical,SubVertical,City Location,Investors Name,InvestmentnType,Remarks,Amount
0,1,2020-01-09,BYJU’S,E-Tech,E-learning,Bengaluru,Tiger Global Management,Private Equity Round,NaN,3.076923e+06
1,2,2020-01-13,Shuttl,Transportation,App based shuttle service,Gurgaon,Susquehanna Growth Equity,Series C,NaN,1.238214e+05
2,3,2020-01-09,Mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,Sequoia Capital India,Series B,NaN,2.824440e+05
3,4,2020-01-02,https://www.wealthbucket.in/,FinTech,Online Investment,New Delhi,Vinod Khatumal,Pre-series A,NaN,4.615385e+04
4,5,2020-01-02,Fashor,Fashion and Apparel,Embroiled Clothes For Women,Mumbai,Sprout Venture Partners,Seed Round,NaN,2.769231e+04


In [13]:
# check outliers in amount column
# --------------------------------

df['Amount'].describe()

count    2.060000e+03
mean     2.841954e+05
std      1.869502e+06
min      2.461538e+02
25%      7.230769e+03
50%      2.692308e+04
75%      1.230769e+05
max      6.000000e+07
Name: Amount, dtype: float64

In [14]:
# Step 1: Calculate IQR (ab USD mein)
Q1 = df['Amount'].quantile(0.25)
Q3 = df['Amount'].quantile(0.75)
IQR = Q3 - Q1

# Step 2: Calculate bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Step 3: Identify outliers
outliers = df[(df['Amount'] < lower_bound) | (df['Amount'] > upper_bound)]
print(f"Number of outliers: {len(outliers)}")

Number of outliers: 283


In [15]:
# Industry Vertical standardization — explicit mapping for the supplied labels
# -----------------------------------------------------------------------------
# One-cell cleaning block: the full mapping is embedded below, so no extra CSV
# or .py file is required. Some descriptions can fit multiple industries; review
# the unmatched labels and category counts before publishing final findings.

import re
import unicodedata

INDUSTRY_MAP = {'E-Tech': 'Technology & Software',
 'Transportation': 'Transportation & Logistics',
 'E-commerce': 'Retail & E-commerce',
 'FinTech': 'Financial Services & FinTech',
 'Fashion and Apparel': 'Fashion, Beauty & Lifestyle',
 'Logistics': 'Transportation & Logistics',
 'Hospitality': 'Travel & Hospitality',
 'Technology': 'Technology & Software',
 'E-Commerce': 'Retail & E-commerce',
 'Aerospace': 'Industrial & Manufacturing',
 'B2B-focused foodtech startup': 'Food & Beverage',
 'Video': 'Media, Entertainment & Social',
 'Gaming': 'Media, Entertainment & Social',
 'Software': 'Technology & Software',
 'Health and wellness': 'Healthcare & Wellness',
 'Education': 'Education & EdTech',
 'Food and Beverage': 'Food & Beverage',
 'Health and Wellness': 'Healthcare & Wellness',
 'B2B Marketing': 'Marketing & Customer Engagement',
 'Video Games': 'Media, Entertainment & Social',
 'SaaS': 'Technology & Software',
 'Last Mile Transportation': 'Transportation & Logistics',
 'Healthcare': 'Healthcare & Wellness',
 'Customer Service': 'Marketing & Customer Engagement',
 'B2B': 'Other / Needs Review',
 'Consumer Goods': 'Industrial & Manufacturing',
 'Advertising, Marketing': 'Marketing & Customer Engagement',
 'IoT': 'Technology & Software',
 'Information Technology': 'Technology & Software',
 'Consumer Technology': 'Technology & Software',
 'Accounting': 'Financial Services & FinTech',
 'Finance': 'Financial Services & FinTech',
 'Customer Service Platform': 'Marketing & Customer Engagement',
 'Automotive': 'Transportation & Logistics',
 'EdTech': 'Education & EdTech',
 'Services': 'Other / Needs Review',
 'Compliance': 'Other / Needs Review',
 'Transport': 'Transportation & Logistics',
 'Artificial Intelligence': 'Technology & Software',
 'Tech': 'Technology & Software',
 'Health Care': 'Healthcare & Wellness',
 'Luxury Label': 'Fashion, Beauty & Lifestyle',
 'Waste Management Service': 'Energy & CleanTech',
 'Deep-Tech': 'Technology & Software',
 'Energy': 'Energy & CleanTech',
 'Digital Media': 'Media, Entertainment & Social',
 'Saas': 'Technology & Software',
 'Automobile': 'Transportation & Logistics',
 'Agtech': 'Agriculture & AgriTech',
 'Social Media': 'Media, Entertainment & Social',
 'Fintech': 'Financial Services & FinTech',
 'Edtech': 'Education & EdTech',
 'AI': 'Technology & Software',
 'Nanotechnology': 'Technology & Software',
 'Services Platform': 'Other / Needs Review',
 'Travel Tech': 'Travel & Hospitality',
 'Online Education': 'Education & EdTech',
 'Online Marketplace': 'Retail & E-commerce',
 'SaaS, Ecommerce': 'Technology & Software',
 'NBFC': 'Financial Services & FinTech',
 'Food': 'Food & Beverage',
 'Food Tech': 'Food & Beverage',
 'Automation': 'Technology & Software',
 'Investment': 'Financial Services & FinTech',
 'Social Network': 'Media, Entertainment & Social',
 'Financial Tech': 'Financial Services & FinTech',
 'Fashion': 'Fashion, Beauty & Lifestyle',
 'Real Estate': 'Real Estate & Home Services',
 'Logistics Tech': 'Transportation & Logistics',
 'Consumer Internet': 'Retail & E-commerce',
 'B2B Platform': 'Other / Needs Review',
 'Fin-Tech': 'Financial Services & FinTech',
 'Food-Tech': 'Food & Beverage',
 'Ed-Tech': 'Education & EdTech',
 'IT': 'Technology & Software',
 'Ed-tech': 'Education & EdTech',
 'Clean-tech': 'Energy & CleanTech',
 'Fiinance': 'Financial Services & FinTech',
 'Food & Beverages': 'Food & Beverage',
 'Media': 'Media, Entertainment & Social',
 'Food and Beverages': 'Food & Beverage',
 'Publishing': 'Media, Entertainment & Social',
 'Agriculture': 'Agriculture & AgriTech',
 'Consumer internet': 'Retail & E-commerce',
 'ECommerce': 'Retail & E-commerce',
 'Inspiration': 'Media, Entertainment & Social',
 'Storytelling': 'Media, Entertainment & Social',
 'Lifestyle': 'Fashion, Beauty & Lifestyle',
 'Food & Beverage': 'Food & Beverage',
 'eCommece': 'Retail & E-commerce',
 'Consumer Portal': 'Retail & E-commerce',
 'Others': 'Other / Needs Review',
 'ecommerce': 'Retail & E-commerce',
 'healthcare': 'Healthcare & Wellness',
 'FMCG': 'Industrial & Manufacturing',
 'Reality': 'Real Estate & Home Services',
 'Auto': 'Transportation & Logistics',
 'BFSI': 'Financial Services & FinTech',
 'Consumer Interne': 'Other / Needs Review',
 'Online Education Information platform': 'Education & EdTech',
 'Brand Licensing Startup': 'Marketing & Customer Engagement',
 'Gourmet Food Discovery & Delivery platform': 'Food & Beverage',
 'Transportation & Logistics Platform': 'Transportation & Logistics',
 'Enterprise Marketing Automation platform': 'Marketing & Customer Engagement',
 'Digital Healthcare': 'Healthcare & Wellness',
 'Last Minute Hotel Booking App': 'Travel & Hospitality',
 'Womens Fashion Wear Portal': 'Fashion, Beauty & Lifestyle',
 'Product Learning platform': 'Education & EdTech',
 'Online Food ordering & Delivery platform': 'Food & Beverage',
 'App based Bus Pooling Services': 'Transportation & Logistics',
 'Social Learning Platform': 'Education & EdTech',
 'Social Fitness platform': 'Healthcare & Wellness',
 'On Demand Mobile app developer': 'Technology & Software',
 'Car Maintenance & Management mobile app': 'Transportation & Logistics',
 'Online Wedding Marketplace': 'Media, Entertainment & Social',
 'Splitting Bills Mobile App': 'Technology & Software',
 'IOT Energy Management Analytics platform': 'Technology & Software',
 'Custom Made furniture e-tailer': 'Retail & E-commerce',
 'QSR Chain': 'Food & Beverage',
 'eLearning platform': 'Education & EdTech',
 'Smart Safety Wearable Devices': 'Technology & Software',
 'Cab Sharing service Mobile app': 'Transportation & Logistics',
 'Cloud Enterprise Mobility Platform': 'Transportation & Logistics',
 'Online Shopping Assistant Mobile app': 'Retail & E-commerce',
 'Home Cooked Food Order & Delivery platform': 'Food & Beverage',
 'Online School for Analytics learning': 'Education & EdTech',
 'Chat based personal Assistant App': 'Technology & Software',
 'Picture creation & Social mobile app': 'Media, Entertainment & Social',
 'Nonbanking finance company': 'Financial Services & FinTech',
 'eCommerce returns etailer': 'Retail & E-commerce',
 'Online Freight Services Aggregator': 'Transportation & Logistics',
 'Online Branded Furniture etailer': 'Retail & E-commerce',
 'Digital Analytics Platform': 'Technology & Software',
 'Internet Network Infrastructure Services': 'Technology & Software',
 'Supply Chain & Logistics Solutions': 'Transportation & Logistics',
 'Holiday Resort Chain': 'Travel & Hospitality',
 'Mobile Point of Sale solutions': 'Financial Services & FinTech',
 'Peer-to-Peer Money Transfer & Recharge App': 'Financial Services & FinTech',
 'Test Automation SAAS platform': 'Technology & Software',
 'Citizens Engagement Platform': 'HR & Professional Services',
 'Indoor Navigation & Analytics Solutions': 'Technology & Software',
 'Hyperlocal On-Demand Household Services platform': 'Real Estate & Home Services',
 'Online RTI application services': 'Other / Needs Review',
 'ECommerce Website Creation SAAS platform': 'Technology & Software',
 'Wedding Venues & Vendors Marketplace': 'Media, Entertainment & Social',
 'Interface development platform for Government officials': 'HR & Professional Services',
 'Modular Furnishings Marketplace': 'Retail & E-commerce',
 'Private Jet Bookings Marketplace': 'Transportation & Logistics',
 'Cab search Comparison & Booking platform': 'Transportation & Logistics',
 'Performance Optimization Platform for athletes': 'Healthcare & Wellness',
 'Event Venue Booking Platform': 'Travel & Hospitality',
 'Digital Media Platform': 'Media, Entertainment & Social',
 'visual search and discovery platform': 'Media, Entertainment & Social',
 'Home services marketplace': 'Real Estate & Home Services',
 'End to End reverse logistics Solution platform': 'Transportation & Logistics',
 'Virtual Health consultation app': 'Healthcare & Wellness',
 'Online Renting platform': 'Real Estate & Home Services',
 'Logistics Services Provider': 'Transportation & Logistics',
 'Data Science & UX design Learning platform': 'Education & EdTech',
 'Online marketplace for Chef Meals': 'Food & Beverage',
 'Electric Bike Manufacturers': 'Industrial & Manufacturing',
 'ECommerce platform solutions': 'Technology & Software',
 'Online Media Publication': 'Media, Entertainment & Social',
 'International Hiring Platform': 'HR & Professional Services',
 'Logistics Service Provider Marketplace': 'Transportation & Logistics',
 'Hyperlocal\\\\xc2\\\\xa0 Grocery Delivery Service': 'Food & Beverage',
 'Sports Education Platform': 'Education & EdTech',
 'Bike, Appliances Renting marketplace': 'Retail & E-commerce',
 'Online Marketplace for Industrial Goods': 'Industrial & Manufacturing',
 'Reward points mobile app': 'Marketing & Customer Engagement',
 'Mobile accessories online store': 'Fashion, Beauty & Lifestyle',
 'Content Discovery & reward points platform': 'Media, Entertainment & Social',
 'Artificial Intelligence platform': 'Technology & Software',
 'Medical Consultation & Doctor appointment booking platform': 'Healthcare & Wellness',
 'Mobile Fitness App': 'Healthcare & Wellness',
 'Digital Coupons, Deal & Cashback aggregator app': 'Marketing & Customer Engagement',
 'Mobile Game development & Design platform': 'Media, Entertainment & Social',
 'B2B eCommerce Marketplace': 'Retail & E-commerce',
 'Furniture and appliances rental platform': 'Real Estate & Home Services',
 'B2B Merchandize platform': 'Fashion, Beauty & Lifestyle',
 'Real Estate focused Tech platform': 'Real Estate & Home Services',
 'Photographer Online search & booking platform': 'Travel & Hospitality',
 'Grocery Delivery platform': 'Food & Beverage',
 'Mobile Car Servicing appointment app': 'Transportation & Logistics',
 'New Curation Mobile App': 'Technology & Software',
 'Mobile Wallet': 'Financial Services & FinTech',
 'Online Food Ordering & Delivery platform': 'Food & Beverage',
 'Bus Aggregation and rental mobile app': 'Transportation & Logistics',
 'hyperlocal online Services platform': 'Other / Needs Review',
 'WiFi \\xe2\\x80\\x93 first Cloud communication platform': 'Technology & Software',
 'Alternate Mobile Monetization platform': 'Financial Services & FinTech',
 'Online Food Ordering Marketplace': 'Food & Beverage',
 'Quick Service Restaurant & Online Delivery': 'Food & Beverage',
 'Budget Hotels Aggregator': 'Travel & Hospitality',
 'Efficient Energy Management platform': 'Technology & Software',
 'Digital Intelligent learning platform': 'Education & EdTech',
 'Voice Call incentivization mobile app': 'Marketing & Customer Engagement',
 'Online Counselling and psychological support platform': 'Healthcare & Wellness',
 'Domestic Help Aggregation platform': 'Technology & Software',
 'Home Stay & room rentals platform': 'Travel & Hospitality',
 'Selfie Mobile App': 'Media, Entertainment & Social',
 'Online Reputation Management Platform': 'Media, Entertainment & Social',
 'Online Grocery platform': 'Food & Beverage',
 'Online content platform for women': 'Media, Entertainment & Social',
 'Online Vehicle Spare Parts etailer': 'Retail & E-commerce',
 'Event Ticketing platform': 'Technology & Software',
 'affordable Personal Healthcare Products': 'Healthcare & Wellness',
 'Premium dining Lounges & restaurants': 'Food & Beverage',
 'Mobile Growth Hacking\\\\nPlatform': 'Marketing & Customer Engagement',
 'Skill Training Startup': 'Education & EdTech',
 'Beauty and Wellness Marketplace': 'Healthcare & Wellness',
 'Marketplace App for Bangkok': 'HR & Professional Services',
 'On-demand Maids Service Provider': 'Real Estate & Home Services',
 'eCommerce Product Search Engine': 'Retail & E-commerce',
 'Ready to cook packaged Foods': 'Food & Beverage',
 'Product Customization Platform': 'Technology & Software',
 'Branded Food products online sales': 'Food & Beverage',
 'Talent platform for Fashion professionals': 'Fashion, Beauty & Lifestyle',
 'Auto Insurance Online platform': 'Financial Services & FinTech',
 'Hyperlocal Logistics Service Provider': 'Transportation & Logistics',
 'Cricket Management Mobile Game': 'Media, Entertainment & Social',
 'On Demand Laundry Service platform': 'Fashion, Beauty & Lifestyle',
 'Ecommerce Discount & Cashback coupons platform': 'Marketing & Customer Engagement',
 'Virtual Reality Headset creator': 'Technology & Software',
 'Home Improvement Service platform': 'Real Estate & Home Services',
 'mobile-only tasks marketplace': 'Retail & E-commerce',
 'Women Ethnic Wear\\\\xc2\\\\xa0 Online Marketplace': 'Fashion, Beauty & Lifestyle',
 'Numerical Computing & Data Science Platform': 'Technology & Software',
 'Designer fashion Jewellery Marketplace': 'Fashion, Beauty & Lifestyle',
 'Online Products Discovery Platform': 'Technology & Software',
 'Kids Tool kit for Innovation platform': 'Education & EdTech',
 'Mobile Learning Solutions': 'Education & EdTech',
 'Car Aggregator & Retail Mobile App': 'Transportation & Logistics',
 'conversational commerce mobile app': 'Technology & Software',
 'Viral Content web Platform': 'Media, Entertainment & Social',
 'Express local delivery platform': 'Transportation & Logistics',
 'On-Demand Local Logistics provider': 'Transportation & Logistics',
 'Curated Freelancer Marketplace': 'HR & Professional Services',
 'Online Taxi Rental Platform': 'Transportation & Logistics',
 'Mobile Services Marketplace': 'Other / Needs Review',
 'Smartwatch Maker': 'Technology & Software',
 'Online Gourmet Food Marketplace': 'Food & Beverage',
 'Online Jewellery etailer': 'Fashion, Beauty & Lifestyle',
 'Sales & Productivity Software': 'Marketing & Customer Engagement',
 'ecommerce related software product platform': 'Technology & Software',
 'Health-Tech platform': 'Healthcare & Wellness',
 'Nightlife Discovery Mobile App': 'Travel & Hospitality',
 'Online Real Estate Marketplace': 'Real Estate & Home Services',
 'Home Healthcare Services platform,': 'Healthcare & Wellness',
 'Indian Ethnic Crafts Etailer': 'Fashion, Beauty & Lifestyle',
 'Social platform for traders and investors': 'Technology & Software',
 'Online Furniture, Home Appliances Rental Platform': 'Real Estate & Home Services',
 'Healthcare Services Discovery platform': 'Healthcare & Wellness',
 'Health and Beauty Services Marketplace': 'Healthcare & Wellness',
 'Automotive Services platform': 'Transportation & Logistics',
 'Online and TV Shopping Marketplace': 'Retail & E-commerce',
 'Big Data Analytics Platform': 'Technology & Software',
 'Mobile App based Loan disbursement platform': 'Financial Services & FinTech',
 'On Demand Laundry Services App': 'Real Estate & Home Services',
 'Education Content Provider': 'Education & EdTech',
 'Content Management Software Solutions': 'Media, Entertainment & Social',
 'Fitness Tracking Management Solutions': 'Healthcare & Wellness',
 'Hyper-local Grocery Delivery platform': 'Food & Beverage',
 'Hyperlocal healthcare product Delivery Service provider': 'Healthcare & Wellness',
 'Online Lending Marketplace': 'Financial Services & FinTech',
 'On Demand Beauty Services Platform': 'Healthcare & Wellness',
 'Education Content etailer': 'Education & EdTech',
 'Clud based Learning platform': 'Education & EdTech',
 'Talent Exchange & Talent Services Marketplace': 'HR & Professional Services',
 'Two-Wheeler Taxi Service': 'Transportation & Logistics',
 'character merchandize sales platform': 'Fashion, Beauty & Lifestyle',
 'Mobile Application Developer': 'Technology & Software',
 'Ecommerce Delivery locker services': 'Transportation & Logistics',
 'extra curricular activities marketplace app': 'Retail & E-commerce',
 'Hyperlocal Online Home services provider': 'Real Estate & Home Services',
 'Easy Business Loans platform': 'Technology & Software',
 'Performance based Wholesale Marketplace': 'Retail & E-commerce',
 'Professional Services Marketplace': 'HR & Professional Services',
 'Movie-on-demand platform': 'Media, Entertainment & Social',
 'us ticketing and fleet management platform': 'Transportation & Logistics',
 'Online Income Tax Filing platform': 'Financial Services & FinTech',
 'Product discovery & recommendation platform': 'Technology & Software',
 'Mobile Payments App': 'Financial Services & FinTech',
 'Beauty & Wellness Products e-tailer': 'Healthcare & Wellness',
 'Fashion jewelry and accessories e-tailer': 'Fashion, Beauty & Lifestyle',
 'Online Consumer Lending platform': 'Financial Services & FinTech',
 'Premium Apparel shopping portal': 'Fashion, Beauty & Lifestyle',
 'Exclusive Platform for Doctors & Healthcare professionals': 'Healthcare & Wellness',
 'Job Board': 'HR & Professional Services',
 'Sports Management & Physical Education Business': 'Education & EdTech',
 'multi-sport program for preschoolers': 'Education & EdTech',
 'QSR & Online delivery portal': 'Food & Beverage',
 'Online Homeopathy Clinic': 'Healthcare & Wellness',
 'E-Learning Service Provider': 'Education & EdTech',
 'Organic Food etailer': 'Food & Beverage',
 'Delivery & Logistics Service provider': 'Transportation & Logistics',
 'Product Discovery & comparison App': 'Retail & E-commerce',
 'Online Apparels Fashion brand': 'Fashion, Beauty & Lifestyle',
 'Home Cooked Food marketplace & Delivery': 'Food & Beverage',
 'Gesture based Mobile Development': 'Technology & Software',
 'Hotel Mobile CRM Software platform': 'Travel & Hospitality',
 'Rental Accommodation Search platform': 'Travel & Hospitality',
 'Logistics Tech Platform': 'Transportation & Logistics',
 'Developer Portfolio Showcase platform': 'Technology & Software',
 'Doctors Network Mobile App': 'Healthcare & Wellness',
 'End-to-End Lending platform': 'Financial Services & FinTech',
 'on-demand healthcare marketplace': 'Healthcare & Wellness',
 '360-degree view creating platform': 'Technology & Software',
 'Advertising & Marketing Community Networking platform': 'Marketing & Customer Engagement',
 'Food Ordering & Delivery App': 'Food & Beverage',
 'Ride Sharing platform': 'Transportation & Logistics',
 'Online P2P lending marketplace': 'Financial Services & FinTech',
 'B2B marketplace for industrial goods': 'Industrial & Manufacturing',
 'Cloud Based Collaboration platform': 'Technology & Software',
 'global community for travellers': 'Media, Entertainment & Social',
 'Prepaid Bill manager App': 'Financial Services & FinTech',
 'Mobile Only Shopping Assistant': 'Retail & E-commerce',
 'Mobile Messaging Assistant App': 'Technology & Software',
 'Personalized Wish List creator app': 'Retail & E-commerce',
 'Music Streaming mobile app': 'Media, Entertainment & Social',
 'Raw Meat & Ready to eat food etailer': 'Food & Beverage',
 'Gym Discovery platform': 'Healthcare & Wellness',
 'Online Purchase rewards app': 'Marketing & Customer Engagement',
 'Home rental platform': 'Real Estate & Home Services',
 'Crowdsourced Delivery platform': 'Transportation & Logistics',
 'Real Estate Insights platform': 'Real Estate & Home Services',
 'Structural & Civil Engg Service Automation': 'HR & Professional Services',
 'Startup Funding Marketplace': 'Financial Services & FinTech',
 'Payments Solution platform': 'Financial Services & FinTech',
 'Travel Destination Discovery platform': 'Travel & Hospitality',
 'Order Fulfillment SAAS platform': 'Transportation & Logistics',
 'Last Mile Delivery Service': 'Transportation & Logistics',
 'Healthcare IT Solutions & services': 'Healthcare & Wellness',
 'Gourmet Meals Delivery': 'Food & Beverage',
 'Healthy Meals Food delivery platform': 'Food & Beverage',
 'Real Estate Mobile CRM': 'Real Estate & Home Services',
 'Micro-Brewery': 'Food & Beverage',
 'Home Made Food Marketplace': 'Food & Beverage',
 'residential rental management platform': 'Real Estate & Home Services',
 'Personalized Stock Intelligence Platform': 'Financial Services & FinTech',
 'Personal Diagnostic Mobile App': 'Healthcare & Wellness',
 'Online food ordering & Delivery service': 'Food & Beverage',
 'Hotel Aggregator & booking platform': 'Travel & Hospitality',
 'Online Logistics Platform': 'Transportation & Logistics',
 'Online Student & Campus Social Networking platform': 'Education & EdTech',
 'Professionals & Project Search Marketplace': 'Retail & E-commerce',
 'Online Pharmacy': 'Healthcare & Wellness',
 'Online Payment Gateway': 'Financial Services & FinTech',
 'Psychometric Test Online Software': 'Education & EdTech',
 'Location based Nightlife recommendation\\\\xc2\\\\xa0 Platform': 'Travel & Hospitality',
 'Professional Health Services Platform': 'Healthcare & Wellness',
 'Hyperlocal Logistics Service': 'Transportation & Logistics',
 'Online user engagement platform': 'Marketing & Customer Engagement',
 'Asset Financing platform': 'Financial Services & FinTech',
 'Virtual Reality activity based learning platform': 'Education & EdTech',
 'Online Grocery Store': 'Food & Beverage',
 'Investment management platform': 'Financial Services & FinTech',
 'Industrial Toons Marketplace': 'Media, Entertainment & Social',
 'Public Commute helper App': 'Transportation & Logistics',
 'Artificial Intelligence Ecommerce Chatbot': 'Technology & Software',
 'Truck Aggregator & Logistics service': 'Transportation & Logistics',
 'Ridesharing Mobile app': 'Transportation & Logistics',
 'Food Discovery & Delivery Mobile app': 'Food & Beverage',
 'Competitive exam learning platform': 'Education & EdTech',
 'Document Digitization platform': 'Technology & Software',
 'proximity marketing & Mobile Advertising platform': 'Marketing & Customer Engagement',
 'Autorickshaw Aggregator & Booking platform': 'Transportation & Logistics',
 'FinTech Startup Incubation platform': 'Financial Services & FinTech',
 'Payment Services platform': 'Financial Services & FinTech',
 'Personalized Styling platform': 'Fashion, Beauty & Lifestyle',
 'Luxury goods\\\\xc2\\\\xa0 Shopping Platform': 'Fashion, Beauty & Lifestyle',
 'Food Delivery Platform': 'Food & Beverage',
 'Online Lingerie Marketplace': 'Fashion, Beauty & Lifestyle',
 'Cloud software solutions': 'Technology & Software',
 'Job Search Platform': 'HR & Professional Services',
 'Doctor consultancy Mobile App': 'Healthcare & Wellness',
 'Small Business Financing (NBFC)': 'Financial Services & FinTech',
 'Beauty & Wellness Services Marketplace': 'Healthcare & Wellness',
 'Online Movie Review Platform': 'Media, Entertainment & Social',
 'ECommerce Brands\\\\xe2\\\\x80\\\\x99 Full Service Agency': 'Retail & E-commerce',
 'Cloud-based Hotel Booking Platform': 'Travel & Hospitality',
 'Community Driven News/Views Platform': 'Media, Entertainment & Social',
 'Study Material Marketplace': 'Education & EdTech',
 'Restaurant Ratings & Reviews platform': 'Food & Beverage',
 'Dairy Based Product Manufacturer': 'Food & Beverage',
 'Government Test Preparation platform': 'Education & EdTech',
 'Smart Report Cards for Schools': 'Education & EdTech',
 'Teacher empowerment platform': 'Education & EdTech',
 'Multilingual Test Preparation Platform': 'Education & EdTech',
 'Industrial Tools Marketplace': 'Industrial & Manufacturing',
 'Non Profit Organization to alleviate poverty': 'HR & Professional Services',
 'Gym & Fitness Studios Subscription platform': 'Healthcare & Wellness',
 'pre-used luxury item Marketplace': 'Retail & E-commerce',
 'Home Design & D\\\\xc3\\\\xa9cor platform': 'Real Estate & Home Services',
 'Used Vehicles Marketplace': 'Transportation & Logistics',
 'Online Bus & Cab ticketing platform': 'Transportation & Logistics',
 'Mobile App testing platform': 'Technology & Software',
 'Hyperlocal Delivery Services': 'Transportation & Logistics',
 'Early Cancer Detection Solutions': 'Healthcare & Wellness',
 'Cab rental Mobile app': 'Transportation & Logistics',
 'Online Insurance Distribution Platform': 'Financial Services & FinTech',
 'Mobile Marketing Automation Platform': 'Marketing & Customer Engagement',
 'Prepaid Mobile Bill Manager App': 'Financial Services & FinTech',
 'Chain of Tea Caf\\\\xc3\\\\xa9\\\\xe2\\\\x80\\\\x99s': 'Food & Beverage',
 'Branded Budget Hotel Marketplace': 'Travel & Hospitality',
 'Home Fitting & Fixtures Marketplace': 'Real Estate & Home Services',
 'Food Subscription platform': 'Food & Beverage',
 'Wedding Planning Platform': 'Technology & Software',
 'B2B Mobile Auction Marketplace': 'Retail & E-commerce',
 'ECommerce Data Analytics Platform': 'Technology & Software',
 'Designer Merchandize Marketplace': 'Fashion, Beauty & Lifestyle',
 'Private label Fashion eTailer': 'Fashion, Beauty & Lifestyle',
 'Online Food ordering & delivery': 'Food & Beverage',
 'Real Estate Broker Platform App': 'Real Estate & Home Services',
 'Online Health & Wellness platform': 'Healthcare & Wellness',
 'Internet of Things platform': 'Technology & Software',
 'Diagnostic Labs aggregator platform': 'Healthcare & Wellness',
 'Hyperlocal Delivery Platform': 'Transportation & Logistics',
 'Home Medical Care Services': 'Healthcare & Wellness',
 'Real-Time stock data platform': 'Financial Services & FinTech',
 'Food discovery & table booking app': 'Food & Beverage',
 'coupons and cashback aggregator app': 'Marketing & Customer Engagement',
 'Financial Products lead generation platform': 'Financial Services & FinTech',
 'Sustainable Agri-inputs Firm': 'Agriculture & AgriTech',
 'Financial Services Portal': 'Financial Services & FinTech',
 'Pre-owned games Marketplace': 'Media, Entertainment & Social',
 'Content-based Subscription eCommerce platform': 'Media, Entertainment & Social',
 'virtual reality, 3d simulation and stereoscopic products': 'Technology & Software',
 'Online Interior Designing platform': 'Real Estate & Home Services',
 'End-to-end Logistics platform': 'Transportation & Logistics',
 'Private Label Apparel sales': 'Fashion, Beauty & Lifestyle',
 'Healthcare Consulting platform': 'Healthcare & Wellness',
 'Stock Market Portal': 'Financial Services & FinTech',
 'Cross-device retargeting platform': 'Marketing & Customer Engagement',
 'Digital & Physical Publishing platform': 'Media, Entertainment & Social',
 'Data Driven Publishing platform': 'Media, Entertainment & Social',
 'Employee OnBoarding & Orientation platform': 'HR & Professional Services',
 'Solar Power Solutions company': 'Energy & CleanTech',
 'Online Meat Ordering platform': 'Food & Beverage',
 'online meal-booking platform for train travelers': 'Food & Beverage',
 'hyperlocal delivery platform': 'Transportation & Logistics',
 'Cab Booking app platform': 'Transportation & Logistics',
 'E-Commerce & M-Commerce platform': 'Technology & Software',
 'Mobile Advertising platform': 'Marketing & Customer Engagement',
 'Hyperlocal Grocery Delivery': 'Food & Beverage',
 'Ecommerce Product recommendation platform': 'Marketing & Customer Engagement',
 'Activity Based Social Network': 'Media, Entertainment & Social',
 'Fashion Discovery platform': 'Fashion, Beauty & Lifestyle',
 'Consumer lending marketplace': 'Financial Services & FinTech',
 'Big Data Management Platform': 'Technology & Software',
 'Budget Accommodation Platform': 'Travel & Hospitality',
 'Local Mobile OS': 'Technology & Software',
 'Digital Signal processing platform': 'Technology & Software',
 'Ecommerce Marketplace': 'Retail & E-commerce',
 'Restaurant Discount app': 'Food & Beverage',
 'Tech-enabled learning': 'Education & EdTech',
 'ECommerce Logistics provider': 'Transportation & Logistics',
 'on-demand home beauty and wellness portal': 'Healthcare & Wellness',
 'Self Driven Car rental': 'Transportation & Logistics',
 'Budget Hotel accommodation brand': 'Travel & Hospitality',
 'Energy & Environment Online Marketplace': 'Energy & CleanTech',
 'User Engagment & Analytics platform': 'Technology & Software',
 'On-Demand Delivery Logistics': 'Transportation & Logistics',
 'fund raising platform for startups': 'Financial Services & FinTech',
 'Online budget hotel chain': 'Travel & Hospitality',
 'Hyperlocal Handyman services': 'Real Estate & Home Services',
 'hotel booking app': 'Travel & Hospitality',
 'Rental Accomodation finder': 'Real Estate & Home Services',
 'Tech Platform for property brokers': 'Real Estate & Home Services',
 'Luxury Apparel rental': 'Fashion, Beauty & Lifestyle',
 'App security\\\\xc2\\\\xa0 testing platform': 'Technology & Software',
 'Doctor Appointment booking app': 'Healthcare & Wellness',
 'anonymous bulletin board app': 'Media, Entertainment & Social',
 'Travel community platform': 'Travel & Hospitality',
 'Crowd funding platform': 'Financial Services & FinTech',
 'Mobile based PoS solution': 'Financial Services & FinTech',
 'Car & Bike ecommerce platform': 'Transportation & Logistics',
 'Fast food Chain Franchisee': 'Food & Beverage',
 'Loyalty Programs & Reward Points': 'Marketing & Customer Engagement',
 'Energy saving solutions provider': 'Energy & CleanTech',
 'Self-Storage Spaces': 'Real Estate & Home Services',
 'Grocery retail Mobile app': 'Food & Beverage',
 'Taxi Rental platform': 'Transportation & Logistics',
 'Machine Learning Platform': 'Education & EdTech',
 'mobile social network': 'Media, Entertainment & Social',
 'Hyper-Local Ecommerce': 'Retail & E-commerce',
 'Online printing marketplace': 'Media, Entertainment & Social',
 'mobile payment software platform': 'Financial Services & FinTech',
 'Automated Storage & Warehousing Solution': 'Transportation & Logistics',
 'Home Design & D\\\\xc3\\\\xa9cor solutions': 'Real Estate & Home Services',
 'Quick Service Restaurant Chain': 'Food & Beverage',
 'Subscription based Online Grocery platform': 'Food & Beverage',
 'engineering services company': 'Industrial & Manufacturing',
 'Business and IT consulting': 'HR & Professional Services',
 'Credit Card Fraud protection solutions': 'Financial Services & FinTech',
 'Online & mobile Grocery store': 'Food & Beverage',
 'Career Community for Women': 'HR & Professional Services',
 'Mutual Fund Investing platform': 'Financial Services & FinTech',
 'Ecommerce Marketing Software Platform': 'Marketing & Customer Engagement',
 'Financial Inclusion platform': 'Financial Services & FinTech',
 'Digital Marketing automation solution': 'Marketing & Customer Engagement',
 'Mobile Commerce for Farmers': 'Agriculture & AgriTech',
 'Picture based Social App': 'Media, Entertainment & Social',
 'Logistics service platform': 'Transportation & Logistics',
 'On-Demand Beauty Services': 'Healthcare & Wellness',
 'Exam Preparation Platform': 'Education & EdTech',
 'Startup Focused Online Publisher': 'Media, Entertainment & Social',
 'Waste Management Solutions': 'Energy & CleanTech',
 'App based Fitness coaching': 'Healthcare & Wellness',
 'On demand cleaning & fixing services': 'Real Estate & Home Services',
 'Beauty & Lifestyle platform': 'Fashion, Beauty & Lifestyle',
 'on-demand delivery service': 'Transportation & Logistics',
 'Data Analytics Platform': 'Technology & Software',
 'Deep Learning Cloud Algorithms': 'Education & EdTech',
 'Location based engagement & rewards platform': 'Marketing & Customer Engagement',
 'Online Identity Verification Services platform': 'Technology & Software',
 'Corporate Wellness App': 'Healthcare & Wellness',
 'Fashion Search & Review Platform': 'Fashion, Beauty & Lifestyle',
 'Hyperlocal Maintenance, Repair & Cleaning services': 'Real Estate & Home Services',
 'pre-owned Luxury online apparel seller': 'Fashion, Beauty & Lifestyle',
 'Online Furniture Marketplace': 'Retail & E-commerce',
 'Online Diagnostic Tests Marketplace': 'Healthcare & Wellness',
 'Online marketplace for Photographers': 'HR & Professional Services',
 'Visual Blogging platform': 'Media, Entertainment & Social',
 'Food Customer analytics platform': 'Food & Beverage',
 'Professional Service Appointment booking service': 'Travel & Hospitality',
 'Scientific Horoscope Online Assistance platform': 'Technology & Software',
 'Tech enabled AC bus service': 'Transportation & Logistics',
 'On-Demand App based Professional service provider': 'HR & Professional Services',
 'Restaurant Reservation Platform': 'Food & Beverage',
 'Cross-channel CRM platform': 'Marketing & Customer Engagement',
 'Professional Custom creators Marketplace': 'HR & Professional Services',
 'on-Demand Washing & Dry-Cleaning': 'Fashion, Beauty & Lifestyle',
 'On-Demand Beauty Service': 'Healthcare & Wellness',
 'solar products and services marketplace': 'Energy & CleanTech',
 'Online Food Community Platform': 'Food & Beverage',
 'Private Cloud Networks SAAS platform': 'Technology & Software',
 'Education Marketplace': 'Education & EdTech',
 'Cloud Based Hotel Sales Platform': 'Travel & Hospitality',
 'online recipe sharing platform': 'Food & Beverage',
 'Beauty & Lifestyle Mobile Marketplace': 'Fashion, Beauty & Lifestyle',
 'QSR Restaurant Chain': 'Food & Beverage',
 'Digital Media publishing platform': 'Media, Entertainment & Social',
 'Online Coffee Delivery platform': 'Food & Beverage',
 'Language Localization Cloud platform': 'Education & EdTech',
 'Big Data & predictive Analysis Platform': 'Technology & Software',
 'Business IT Intelligence Services': 'Technology & Software',
 'CRM / Analytics platform': 'Marketing & Customer Engagement',
 'Ed-Tech Platform': 'Education & EdTech',
 'IT infrastructure & Data Center services': 'Technology & Software',
 'Online Financial Services': 'Financial Services & FinTech',
 'Used Car Marketplace': 'Transportation & Logistics',
 'Language Learning App': 'Education & EdTech',
 'Mobile Fitness Marketplace': 'Healthcare & Wellness',
 'Custom Merchandize platform': 'Fashion, Beauty & Lifestyle',
 'P2P Pre-owned goods marketplace': 'Retail & E-commerce',
 'Custom Furniture Marketplace': 'Retail & E-commerce',
 'Online Tailoring Services': 'Fashion, Beauty & Lifestyle',
 'Rail Ticket Confirmation predictor': 'Transportation & Logistics',
 'Ethnic Beverages manufacturer': 'Food & Beverage',
 'Hyperlocal Goods marketplace': 'Retail & E-commerce',
 'Budget Accommodation aggregator': 'Travel & Hospitality',
 'Real Estate Rating & Analysis': 'Real Estate & Home Services',
 'Food Delivery platform': 'Food & Beverage',
 'Food Discovery App': 'Food & Beverage',
 'Music Streaming App': 'Media, Entertainment & Social',
 'Big Data & Analytics platform': 'Technology & Software',
 'Coupon Aggregator Platform': 'Marketing & Customer Engagement',
 'Electric Scooter manufacturer': 'Industrial & Manufacturing',
 'Online Food-Tech Platform': 'Food & Beverage',
 'Celebrity Fashion Brand': 'Fashion, Beauty & Lifestyle',
 'Online Finance lending platform': 'Financial Services & FinTech',
 'Healthcare Mobile App': 'Healthcare & Wellness',
 'Self-driven vehicle rental': 'Transportation & Logistics',
 '\\\\xc2\\\\xa0News Aggregator mobile app': 'Media, Entertainment & Social',
 '\\\\xc2\\\\xa0Online Jewellery Store': 'Fashion, Beauty & Lifestyle',
 '\\\\xc2\\\\xa0Fashion Info Aggregator App': 'Fashion, Beauty & Lifestyle',
 '\\\\xc2\\\\xa0Online Study Notes Marketplace': 'Retail & E-commerce',
 '\\\\xc2\\\\xa0Pre-School Chain': 'Education & EdTech',
 '\\\\xc2\\\\xa0Premium Loyalty Rewards Point Management': 'Marketing & Customer Engagement',
 '\\\\xc2\\\\xa0Contact Center Software Platform': 'Technology & Software',
 '\\\\xc2\\\\xa0Casual Dining restaurant Chain': 'Food & Beverage',
 '\\\\xc2\\\\xa0Online Grocery Delivery': 'Food & Beverage',
 'Online comparison engine': 'Marketing & Customer Engagement',
 'Residential Rental management platform': 'Real Estate & Home Services',
 'Physical Storage warehouses': 'Industrial & Manufacturing',
 'Tech Recruitment platform': 'HR & Professional Services',
 'Bike Rental Platform': 'Transportation & Logistics',
 'Tech enabled logistics platform': 'Transportation & Logistics',
 'Budget Hotel Accommodation': 'Travel & Hospitality',
 'Restaurant customer engagement platform': 'Food & Beverage',
 'Live video streaming app': 'Media, Entertainment & Social',
 'Hyper local logistics platform': 'Transportation & Logistics',
 'Crowd Funding Platform': 'Financial Services & FinTech',
 'F&B sector Mobile Payment platform': 'Financial Services & FinTech',
 'Industrial Supplies B2B ecommerce': 'Industrial & Manufacturing',
 'Payment Solutions platform': 'Financial Services & FinTech',
 'Online Hiring Platform': 'HR & Professional Services',
 'Organic Food ecommerce': 'Food & Beverage',
 'Fine Dining Restaurant Chain': 'Food & Beverage',
 'Luxury Rental Homes': 'Travel & Hospitality',
 'Mobile Point of Sale payment solution': 'Financial Services & FinTech',
 'Online Grocers': 'Food & Beverage',
 'Casual Dating App': 'Media, Entertainment & Social',
 'Clinical Genomics Provider': 'Healthcare & Wellness',
 'Travel information portal': 'Travel & Hospitality',
 'Fitness Marketplace': 'Healthcare & Wellness',
 'Online & Mobile classified listings': 'Media, Entertainment & Social',
 'Private Jet/ Helicopter Marketplace': 'Transportation & Logistics',
 'Real Estate Broker network App': 'Real Estate & Home Services',
 'Ethnic/ Traditional Fashion Store': 'Fashion, Beauty & Lifestyle',
 'Used Bikes Marketplace': 'Retail & E-commerce',
 'Spa & Salon Management Software': 'Healthcare & Wellness',
 'B2B logistics delivery platform': 'Transportation & Logistics',
 'Logistics Automation Platform': 'Transportation & Logistics',
 'Online home d\\\\xc3\\\\xa9cor marketplace': 'Retail & E-commerce',
 'Hyperlocal food & grocery store': 'Food & Beverage',
 'Hyperlocal Handyman Services': 'Real Estate & Home Services',
 'Medical Tech Instruments': 'Healthcare & Wellness',
 'Hyperlocal Deals Marketplace': 'Retail & E-commerce',
 'Innovative Water Solutions': 'Energy & CleanTech',
 'Online Jewelry Store': 'Fashion, Beauty & Lifestyle',
 'Business intelligence & Analytics': 'Technology & Software',
 'Used Vehicle Marketplace': 'Transportation & Logistics',
 'Educational Board Games': 'Education & EdTech',
 'Chinese food delivery': 'Food & Beverage',
 'On-Demand Laundry & Dry cleaning App': 'Fashion, Beauty & Lifestyle',
 'On-Demand Car services App': 'Transportation & Logistics',
 'Hyperlocal Grocery App': 'Food & Beverage',
 'Online Curated Holiday package bookings': 'Travel & Hospitality',
 'Hasbro Toy Manufacturer': 'Industrial & Manufacturing',
 'Beauty and Wellness Platform': 'Healthcare & Wellness',
 'Healthy Food Manufacturer': 'Food & Beverage',
 'Online Learning Platform': 'Education & EdTech',
 'Healthy Food Online Community': 'Food & Beverage',
 'Used two-wheeler Marketplace': 'Transportation & Logistics',
 'Online Fashion Video Portal': 'Fashion, Beauty & Lifestyle',
 'Probiotic Technology Products Manufacturer': 'Healthcare & Wellness',
 'Architectural Design & Consulting': 'HR & Professional Services',
 'Big Data & Analytics Services': 'Technology & Software',
 'Preventive Healthcare Services': 'Healthcare & Wellness',
 'Tech-enabled Rural Healthcare Services': 'Healthcare & Wellness',
 'Engineering Innovations': 'Industrial & Manufacturing',
 'Real Estate Intelligence Platform': 'Real Estate & Home Services',
 'Online Car Rental Affiliates': 'Transportation & Logistics',
 'Data Center Software platform': 'Technology & Software',
 'Online City & Lifestyle Guide': 'Travel & Hospitality',
 'P2P Payments platform': 'Financial Services & FinTech',
 'Bespoke Merchandize Marketplace': 'Fashion, Beauty & Lifestyle',
 'Women Lifestyle Marketplace': 'Fashion, Beauty & Lifestyle',
 'Logistics Solution Provider': 'Transportation & Logistics',
 'Video Intelligence Platform': 'Media, Entertainment & Social',
 'Hyperlocal Shopping App': 'Retail & E-commerce',
 'eCommerce platform': 'Technology & Software',
 'Healthcare\\\\nService provider': 'Healthcare & Wellness',
 'Auto Rickshaw Based Services': 'Transportation & Logistics',
 'Artist / Designer Marketplace': 'Fashion, Beauty & Lifestyle',
 'Off-grid Solar Power': 'Energy & CleanTech',
 'Private Label lingerie Ecommerce': 'Fashion, Beauty & Lifestyle',
 'Online Food Ordering': 'Food & Beverage',
 'Mobile Commerce Platform': 'Technology & Software',
 'Online Printing Services': 'Media, Entertainment & Social',
 'Digital Marketing Services': 'Marketing & Customer Engagement',
 'Travel Guide Mobile App': 'Travel & Hospitality',
 'Fashion Ecommerce store': 'Fashion, Beauty & Lifestyle',
 'Mobile Compression Software': 'Technology & Software',
 'Peer to Peer Lending platform': 'Financial Services & FinTech',
 'Global Healthcare products': 'Healthcare & Wellness',
 'Home Furnishing Solutions': 'Real Estate & Home Services',
 'Skill Training & Placement Platform': 'Education & EdTech',
 'Intra-City Logistics service': 'Transportation & Logistics',
 'online tiffin service aggregator': 'Food & Beverage',
 'Electronic Goods recycling service': 'Energy & CleanTech',
 'Online Coupon & comparison platform': 'Marketing & Customer Engagement',
 'Parents & Kids Online platform': 'Technology & Software',
 'Dental Clinic chain': 'Healthcare & Wellness',
 'On-Demand Logistics Service': 'Transportation & Logistics',
 '3D Printer Manufacturer': 'Technology & Software',
 'IT / Customer Engagement Consulting': 'Marketing & Customer Engagement',
 'Health & Fitness App': 'Healthcare & Wellness',
 'Online Education Marketplace': 'Education & EdTech',
 'Financial Services Platform': 'Financial Services & FinTech',
 'Social Network for Artists': 'Media, Entertainment & Social',
 'Startup funding platform': 'Technology & Software',
 'Data Driven recruitment recommendation': 'HR & Professional Services',
 'Travel Search Engine': 'Travel & Hospitality',
 'Brain Games Platform': 'Media, Entertainment & Social',
 'Video Advertisement Platform': 'Media, Entertainment & Social',
 'Algorithmic Match Making Platform': 'Technology & Software',
 'Mobile Chat based concierge service': 'Travel & Hospitality',
 'Online Budget Hotel Booking Service': 'Travel & Hospitality',
 'Online education marketplace': 'Education & EdTech',
 'Preventive healthcare services': 'Healthcare & Wellness',
 'Science Learning App': 'Education & EdTech',
 'Beauty and Wellness platform': 'Healthcare & Wellness',
 'Online Education Platform': 'Education & EdTech',
 'SaaS product intelligence platform': 'Technology & Software',
 'App Analytics platform': 'Technology & Software',
 'Micro Finance Platform': 'Financial Services & FinTech',
 'Freight logistics platform': 'Transportation & Logistics',
 'Credit Management Services': 'Financial Services & FinTech',
 'Hyperlocal services marketplace': 'Retail & E-commerce',
 'Beauty services Mobile Marketplace': 'Healthcare & Wellness',
 'Marketing / Advertising Automation platform': 'Marketing & Customer Engagement',
 'Online Lingerie platform': 'Fashion, Beauty & Lifestyle',
 'Railways Information Mobile app': 'Technology & Software',
 'Hyperlocal grocery delivery': 'Food & Beverage',
 'Flat rental Mobile App': 'Real Estate & Home Services',
 'Wealth Management Platform': 'Financial Services & FinTech',
 'Online Food Delivery': 'Food & Beverage',
 'Fund Raising Platform': 'Financial Services & FinTech',
 'Workforce Management Software': 'HR & Professional Services',
 'Online Art Marketplace': 'Media, Entertainment & Social',
 'Affordable Education': 'Education & EdTech',
 'Career Development': 'Education & EdTech',
 'Real Estate Advisory': 'Real Estate & Home Services',
 'Banking Analytics Platform': 'Financial Services & FinTech',
 'Customer Engagement Platform': 'Marketing & Customer Engagement',
 'Ecommerce Logistics': 'Transportation & Logistics',
 'Speech Recognition Solutions': 'Technology & Software',
 'Online Product Comparison': 'Marketing & Customer Engagement',
 'Online Security Platform': 'Technology & Software',
 'Online Medical Diagnostic': 'Healthcare & Wellness',
 'Cloud Data Integration Platform': 'Technology & Software',
 'Mobile Laundry Cleaning Service': 'Fashion, Beauty & Lifestyle',
 'Service-On-Demand Mobile App': 'Technology & Software',
 'Intelligent Data Analytics': 'Technology & Software',
 'Online Office Rental': 'Real Estate & Home Services',
 'Lifestyle Ecommerce Portal': 'Fashion, Beauty & Lifestyle',
 'Online Fashion Aggregator': 'Fashion, Beauty & Lifestyle',
 'Online Job skills Showcase': 'HR & Professional Services',
 'Online News Media': 'Media, Entertainment & Social',
 'Enterprise Communication Platform': 'Technology & Software',
 'Offline Tea Chain': 'Food & Beverage',
 'Business Messaging App': 'Technology & Software',
 'Health Mobile App': 'Healthcare & Wellness',
 'Online Ed-Tech Platform': 'Education & EdTech',
 'Online Travel Planning': 'Travel & Hospitality',
 'Hyperlocal Mobile Marketplace': 'Retail & E-commerce',
 'Trucking Logistic Operations': 'Transportation & Logistics',
 'B2C Messaging App': 'Media, Entertainment & Social',
 'Rooftop Solar Plants': 'Energy & CleanTech',
 'Online Car Portal': 'Transportation & Logistics',
 'Online Table Reservation': 'Food & Beverage',
 'Parents focused Web Content': 'Media, Entertainment & Social',
 'Payments Solution Provider': 'Financial Services & FinTech',
 'On-demand Service marketplace': 'HR & Professional Services',
 'On-Demand Business messaging': 'Technology & Software',
 'Online Home D\\\\xc3\\\\xa9cor': 'Retail & E-commerce',
 'API Workflow platform': 'Technology & Software',
 'Affordable Hotel Booking Online': 'Travel & Hospitality',
 'Logistics Intelligence': 'Transportation & Logistics',
 'Last Mile Logistics': 'Transportation & Logistics',
 'Self-driven Car rental': 'Transportation & Logistics',
 'Weight Management Service': 'Healthcare & Wellness',
 'Electric Scooter Manufacturer': 'Transportation & Logistics',
 'Web Content Publishing': 'Media, Entertainment & Social',
 'Video Streaming': 'Media, Entertainment & Social',
 'Price Comparison': 'Marketing & Customer Engagement',
 'Online Classifieds': 'Media, Entertainment & Social',
 'Travel Entertainment': 'Travel & Hospitality',
 'Digital / Mobile Wallet': 'Financial Services & FinTech',
 'Cab Aggregator': 'Transportation & Logistics',
 'Device Repair Svcs': 'Other / Needs Review',
 'Premium Beverages': 'Food & Beverage',
 'Online Furnishing\\\\neCommerce': 'Real Estate & Home Services',
 'Interactive\\\\xc2\\\\xa0 How-To Guides': 'Media, Entertainment & Social',
 'Mobile Health Tech': 'Healthcare & Wellness',
 'Restaurant Reviews': 'Food & Beverage',
 'Online Grocer': 'Food & Beverage',
 'Online Hotel Booking': 'Travel & Hospitality',
 'Online Kitchen Furniture': 'Retail & E-commerce',
 'Hyper-local Online Services': 'Other / Needs Review',
 'Hyper-Local Online/Mobile Grocery': 'Food & Beverage',
 'Online Policy Aggregator': 'Financial Services & FinTech',
 'Hyperlocal Mobile Services Marketplace': 'Technology & Software',
 'OnDemand\\\\nMobile Handyman Services': 'Real Estate & Home Services',
 'Mobile App Development': 'Technology & Software',
 'On-Demand Handyman Services': 'Real Estate & Home Services',
 'Beauty & Wellness Mobile App': 'Healthcare & Wellness',
 'Material Collection & Recycling': 'Energy & CleanTech',
 'Online Pharmacy & Drug DB': 'Healthcare & Wellness',
 'Customer Support': 'Marketing & Customer Engagement',
 'Online Classifieds & Listings': 'Media, Entertainment & Social',
 'Fashion ECommerce': 'Fashion, Beauty & Lifestyle',
 'Private Coaching Centers': 'Education & EdTech',
 'Online Certification Courses': 'Education & EdTech',
 'Ethnic Product eCommerce': 'Fashion, Beauty & Lifestyle',
 'Last Minute Travel Deals mobile marketplace': 'Travel & Hospitality',
 'usiness expense management': 'Financial Services & FinTech',
 'Auto Rickshaw based Logistics': 'Transportation & Logistics',
 'Restaurant reservation app': 'Food & Beverage',
 'Spam Call block App': 'Marketing & Customer Engagement',
 'Online Insurance Aggregator': 'Financial Services & FinTech',
 'Restaurant Management Platform': 'Food & Beverage',
 'Food Logistics & Delivery': 'Food & Beverage',
 'Interactive Educational Games': 'Education & EdTech',
 'Clean Tech': 'Energy & CleanTech',
 'Grey collar Job Board': 'HR & Professional Services',
 'Online Furniture ecommerce': 'Retail & E-commerce',
 'Robotic Program learning': 'Education & EdTech',
 'Tele-Shopping / eCommerce': 'Retail & E-commerce',
 'Used Gadgets Buy / Refurbishing': 'Retail & E-commerce',
 'Cloud Enterprise Software': 'Technology & Software',
 'Startup Analytics platform': 'Technology & Software',
 'Mobile Food Ordering app': 'Food & Beverage',
 'Financial Markets Software': 'Financial Services & FinTech',
 'Hiring Analytics platform': 'HR & Professional Services',
 'Online Travel Marketplace': 'Travel & Hospitality',
 'Unknown': 'Unknown'}

# Normalize text only for matching (trim spaces, case and encoding artifacts).
def _normalize_industry_key(value):
    if pd.isna(value):
        return None

    value = unicodedata.normalize("NFKC", str(value))
    value = value.replace("\u00a0", " ").replace("\xa0", " ")
    # Some source values store escaped special characters as literal text.
    value = re.sub(r"\\+xc2\\+xa0", " ", value, flags=re.IGNORECASE)
    value = re.sub(r"\\+n", " ", value, flags=re.IGNORECASE)
    value = re.sub(r"\s+", " ", value).strip().casefold()
    return value

# Standardize Industry Vertical
industry_raw = df["Industry Vertical"].copy()
industry_keys = industry_raw.map(_normalize_industry_key)
normalized_industry_map = {
    _normalize_industry_key(original): standardized
    for original, standardized in INDUSTRY_MAP.items()
}

mapped_industry = industry_keys.map(normalized_industry_map)
df["Industry Vertical"] = mapped_industry.where(mapped_industry.notna(), industry_raw)

# Audit unmatched values instead of silently discarding them.
unmapped_mask = industry_raw.notna() & ~industry_keys.isin(normalized_industry_map)
unmapped_values = industry_raw.loc[unmapped_mask].value_counts(dropna=False)

print(f"Industry Vertical unique values before: {industry_raw.nunique(dropna=True)}")
print(f"Industry Vertical unique values after:  {df['Industry Vertical'].nunique(dropna=True)}")
print(f"Unmapped non-null rows: {int(unmapped_mask.sum())}")
if unmapped_values.empty:
    print("All non-null Industry Vertical labels matched the mapping.")
else:
    print("\nLabels not found in the mapping (review these):")
    print(unmapped_values.to_string())

print("\nStandardized category counts:")
print(df["Industry Vertical"].value_counts(dropna=False).to_string())

# Standardize Investment Type using an explicit replacement map.
# Funding stages and funding instruments are kept distinct where their meanings differ.
if "Investment Type" not in df.columns and "InvestmentnType" in df.columns:
    df.rename(columns={"InvestmentnType": "Investment Type"}, inplace=True)

INVESTMENT_TYPE_MAP = {
    # Series stages
    "Series A": "Series A",
    "Series B": "Series B",
    "Series B (Extension)": "Series B Extension",
    "Series C": "Series C",
    "Series D": "Series D",
    "Series E": "Series E",
    "Series F": "Series F",
    "Series G": "Series G",
    "Series H": "Series H",
    "Series J": "Series J",
    "Pre-series A": "Pre-Series A",
    "Pre-Series A": "Pre-Series A",
    "pre-Series A": "Pre-Series A",
    "Pre Series A": "Pre-Series A",

    # Seed and angel funding
    "Seed": "Seed",
    "Seed Round": "Seed",
    "Seed Funding": "Seed",
    "Seed funding": "Seed",
    "Seed\\nFunding": "Seed",
    "Angel": "Angel",
    "Angel Round": "Angel",
    "Angel Funding": "Angel",
    "Seed/ Angel Funding": "Seed / Angel",
    "Seed / Angel Funding": "Seed / Angel",
    "Seed/Angel Funding": "Seed / Angel",
    "Angel / Seed Funding": "Seed / Angel",
    "Seed / Angle Funding": "Seed / Angel",  # source typo: Angle -> Angel

    # Venture and private equity
    "Venture": "Venture",
    "Venture Round": "Venture",
    "Venture - Series Unknown": "Venture - Series Unknown",
    "Single Venture": "Single Venture",
    "Private Equity Round": "Private Equity",
    "Private Equity": "Private Equity",
    "PrivateEquity": "Private Equity",
    "Private\\nEquity": "Private Equity",
    "Private Funding": "Private Funding",
    "Private": "Private (Unspecified)",
    "Corporate Round": "Corporate Round",
    "Maiden Round": "Maiden Round",
    "Bridge Round": "Bridge Round",
    "Funding Round": "Funding Round (Unspecified)",
    "Inhouse Funding": "In-house Funding",
    "Crowd funding": "Crowdfunding",
    "Crowd Funding": "Crowdfunding",

    # Debt and other capital instruments
    "Debt Funding": "Debt Funding",
    "Debt-Funding": "Debt Funding",
    "Debt": "Debt Funding",
    "Debt and Preference capital": "Debt & Preference Capital",
    "Structured Debt": "Structured Debt",
    "Term Loan": "Term Loan",
    "Mezzanine": "Mezzanine",
    "Equity": "Equity",
    "Equity Based Funding": "Equity-Based Funding",
}

def _normalize_investment_type_key(value):
    """Normalize formatting only, for reliable dictionary matching."""
    if pd.isna(value):
        return None
    value = unicodedata.normalize("NFKC", str(value))
    value = value.replace("\u00a0", " ").replace("\xa0", " ")
    # Handle source strings containing literal escaped newline characters.
    value = re.sub(r"\\+n", " ", value, flags=re.IGNORECASE)
    value = value.replace("\n", " ")
    value = re.sub(r"\s+", " ", value).strip().casefold()
    return value

if "Investment Type" in df.columns:
    investment_raw = df["Investment Type"].copy()
    investment_keys = investment_raw.map(_normalize_investment_type_key)
    normalized_investment_map = {
        _normalize_investment_type_key(original): standardized
        for original, standardized in INVESTMENT_TYPE_MAP.items()
    }
    mapped_investment = investment_keys.map(normalized_investment_map)
    df["Investment Type"] = mapped_investment.where(
        mapped_investment.notna(), investment_raw
    )

    # Audit values not covered by the supplied mapping.
    unmapped_investment_mask = (
        investment_raw.notna()
        & ~investment_keys.isin(normalized_investment_map)
    )
    unmapped_investment_values = investment_raw.loc[
        unmapped_investment_mask
    ].value_counts(dropna=False)

    print(f"\nInvestment Type unique values before: {investment_raw.nunique(dropna=True)}")
    print(f"Investment Type unique values after:  {df['Investment Type'].nunique(dropna=True)}")
    print(f"Unmapped Investment Type rows: {int(unmapped_investment_mask.sum())}")
    if unmapped_investment_values.empty:
        print("All non-null Investment Type labels matched the mapping.")
    else:
        print("\nInvestment Type labels not found in the mapping (review these):")
        print(unmapped_investment_values.to_string())

    print("\nStandardized Investment Type counts:")
    print(df["Investment Type"].value_counts(dropna=False).to_string())
else:
    print("\nWarning: no Investment Type column found; skipped its standardization.")


Industry Vertical unique values before: 819
Industry Vertical unique values after:  26
Unmapped non-null rows: 9

Labels not found in the mapping (review these):
Industry Vertical
Retail                                    1
Entertainment                             1
Health, Wellness & Beauty Services App    1
Online Loans Marketplace                  1
Self Driven Rental Car Platform           1
pre-used apparel shopping mobile app      1
Renewable energy solutions                1
Location Based App                        1
Online / Mobile Customer Support          1

Standardized category counts:
Industry Vertical
Retail & E-commerce                       1284
Technology & Software                      592
NaN                                        169
Healthcare & Wellness                      145
Financial Services & FinTech               144
Transportation & Logistics                 138
Food & Beverage                            129
Education & EdTech                          97

In [16]:
# Step 1: Check unique city names
print(df['City  Location'].unique())

# Step 2: Standardize (example)
df['City  Location'] = df['City  Location'].replace({
    'Bengaluru': 'Bangalore',
    'Gurugram': 'Gurgaon',
    'New Delhi': 'New Delhi',
    'Delhi': 'New Delhi',
    'Mumbai/Bengaluru': 'Mumbai',
    # ... aapko saari inconsistencies identify karni hain
})

['Bengaluru' 'Gurgaon' 'New Delhi' 'Mumbai' 'Chennai' 'Pune' 'Noida'
 'Faridabad' 'San Francisco' 'San Jose,' 'Amritsar' 'Delhi' 'Kormangala'
 'Tulangan' 'Hyderabad' 'Burnsville' 'Menlo Park' 'Gurugram' 'Palo Alto'
 'Santa Monica' 'Singapore' 'Taramani' 'Andheri' 'Chembur' 'Nairobi'
 'Haryana' 'New York' 'Karnataka' 'Mumbai/Bengaluru' 'Bhopal'
 'Bengaluru and Gurugram' 'India/Singapore' 'Jaipur' 'India/US' 'Nagpur'
 'Indore' 'New York, Bengaluru' 'California' 'India' 'Ahemadabad'
 'Rourkela' 'Srinagar' 'Bhubneswar' 'Chandigarh' 'Delhi & Cambridge'
 'Kolkatta' 'Kolkata' 'Coimbatore' 'Bangalore' 'Udaipur' nan 'Ahemdabad'
 'Bhubaneswar' 'Ahmedabad' 'Surat' 'Goa' 'Uttar Pradesh' 'Nw Delhi' 'Gaya'
 'Vadodara' 'Trivandrum' 'Missourie' 'Panaji' 'Gwalior' 'Karur' 'Udupi'
 'Kochi' 'Agra' 'Bangalore/ Bangkok' 'Hubli' 'Kerala' 'Kozhikode' 'US'
 'Siliguri' 'USA' 'Lucknow' 'Kanpur' 'SFO / Bangalore' 'London'
 'Seattle / Bangalore' 'Pune/Seattle' 'Pune / Dubai' 'Bangalore / SFO'
 'Varanasi' 'New Del

In [17]:
# count values
# -------------

print(df['Industry Vertical'].value_counts())
print("\n")
print(df['City  Location'].value_counts())

Industry Vertical
Retail & E-commerce                       1284
Technology & Software                      592
Healthcare & Wellness                      145
Financial Services & FinTech               144
Transportation & Logistics                 138
Food & Beverage                            129
Education & EdTech                          97
Media, Entertainment & Social               66
Fashion, Beauty & Lifestyle                 51
Real Estate & Home Services                 48
Travel & Hospitality                        42
Marketing & Customer Engagement             39
HR & Professional Services                  29
Other / Needs Review                        20
Industrial & Manufacturing                  15
Energy & CleanTech                          15
Agriculture & AgriTech                       5
Retail                                       1
Entertainment                                1
Health, Wellness & Beauty Services App       1
Online Loans Marketplace                  

In [18]:
# check null values overall data
# -------------------------------

df.isnull().sum()

Sr No                   0
Date                    0
Startup Name            0
Industry Vertical     169
SubVertical           929
City  Location        178
Investors Name         24
Investment Type         4
Remarks              2618
Amount                977
dtype: int64

In [19]:
# Safe fallback: rename only if the source column has not already been renamed.
if "Investment Type" not in df.columns and "InvestmentnType" in df.columns:
    df = df.rename(columns={"InvestmentnType": "Investment Type"})


In [20]:
# Handling null values
# ----------------------

# Step 1: Drop Remarks column (88% missing)
# ------------------------------------------
df = df.drop('Remarks', axis=1)

# Step 2: Fill SubVertical with 'Unknown'
# -----------------------------------------
df['SubVertical'] = df['SubVertical'].fillna('Unknown')

# Step 3: Fill Industry Vertical with 'Unknown'
# ------------------------------------------------
df['Industry Vertical'] = df['Industry Vertical'].fillna('Unknown')

# fill amount rows with Sector-wise median
# -------------------------------------------
df['Amount'] = df['Amount'].fillna(df['Amount'].median())

# Step 5: Drop rows with missing City Location (8.6%)
# -----------------------------------------------------
df['City  Location'] = df['City  Location'].fillna('Unknown')

# Step 6: Drop rows with missing Investors Name (1.2%)
# ------------------------------------------------------
df = df.dropna(subset=['Investors Name'])

# Step 7: Drop rows with missing Investment Type (0.2%)
# ------------------------------------------------------
df = df.dropna(subset=['Investment Type'])

# Step 8: Check remaining missing values
# ----------------------------------------
print(df.isnull().sum())

Sr No                0
Date                 0
Startup Name         0
Industry Vertical    0
SubVertical          0
City  Location       0
Investors Name       0
Investment Type      0
Amount               0
dtype: int64


In [21]:
# Check Duplicates rows
# ----------------------

df.duplicated().sum()

np.int64(0)

In [22]:
df.head()

,Sr No,Date,Startup Name,Industry Vertical,SubVertical,City Location,Investors Name,Investment Type,Amount
0,1,2020-01-09,BYJU’S,Technology & Software,E-learning,Bangalore,Tiger Global Management,Private Equity,3.076923e+06
1,2,2020-01-13,Shuttl,Transportation & Logistics,App based shuttle service,Gurgaon,Susquehanna Growth Equity,Series C,1.238214e+05
2,3,2020-01-09,Mamaearth,Retail & E-commerce,Retailer of baby and toddler products,Bangalore,Sequoia Capital India,Series B,2.824440e+05
3,4,2020-01-02,https://www.wealthbucket.in/,Financial Services & FinTech,Online Investment,New Delhi,Vinod Khatumal,Pre-Series A,4.615385e+04
4,5,2020-01-02,Fashor,"Fashion, Beauty & Lifestyle",Embroiled Clothes For Women,Mumbai,Sprout Venture Partners,Seed,2.769231e+04


In [23]:
df['Industry Vertical'].unique()

array(['Technology & Software', 'Transportation & Logistics',
       'Retail & E-commerce', 'Financial Services & FinTech',
       'Fashion, Beauty & Lifestyle', 'Travel & Hospitality',
       'Industrial & Manufacturing', 'Food & Beverage',
       'Media, Entertainment & Social', 'Healthcare & Wellness',
       'Education & EdTech', 'Marketing & Customer Engagement',
       'Other / Needs Review', 'Energy & CleanTech',
       'Agriculture & AgriTech', 'Real Estate & Home Services',
       'HR & Professional Services', 'Online / Mobile Customer Support',
       'Unknown'], dtype=object)

In [24]:
# Save the file
# ---------------

df.to_csv('cleaned_startup_funding.csv', index=False)

print('File saved successfully...')

File saved successfully...


In [ ]:
# City location and startup names ko bhi clean karna hai